In [2]:
import pandas as pd  
import numpy as np  
import pandas_datareader as pddr  
import yfinance as yf  
from yahoofinancials import YahooFinancials  
import datetime as dt 
import matplotlib.pyplot as plt  
from platform import python_version
from pathlib import Path
# 导入 Path，用于创建输出文件夹和管理文件路径。

In [3]:
# PART 1: 设置参数与输出目录

In [4]:
today = pd.Timestamp.today().normalize()
# 获取运行程序当天的日期，并把时分秒设置为零。

start_date = today - pd.DateOffset(years=1)
# 将开始日期设置为今天往前推一年。

download_start = start_date - pd.Timedelta(days=7)
# 下载时额外提前七天，为节假日和非交易日预留数据。

download_end = today + pd.Timedelta(days=1)
# 将下载结束边界设置为明天，因为 yfinance 的 end 参数不包含当天。

output_dir = Path("asset_analysis")
# 指定主输出文件夹，所有结果都将保存在这里。

data_dir = output_dir / "csv_data"
# 指定保存各资产 CSV 数据的子文件夹。

single_dir = output_dir / "single_asset_charts"
# 指定保存单资产折线图的子文件夹。

comparison_dir = output_dir / "comparison_charts"
# 指定保存六种资产归一化比较图的子文件夹。

for folder in [data_dir, single_dir, comparison_dir]:
    # 依次处理三个输出文件夹。
    folder.mkdir(parents=True, exist_ok=True)
    # 如果文件夹不存在就创建；如果已经存在则不报错。


In [5]:
# PART 2: 定义六种资产和 Yahoo Finance 股票代码

In [6]:
assets = {
    "Bitcoin": ["BTC-USD"],
    "Ethereum": ["ETH-USD"],
    "SP500": ["^GSPC"],
    "Nasdaq100": ["^NDX"],
    "CSI300": ["000300.SS", "399300.SZ"],
    "CSI1000": ["000852.SS", "399852.SZ"]
}
# 字典的键是资产名称，值是对应的 Yahoo Finance 代码候选列表。
# 中国指数提供备用代码；主代码无法下载时才尝试备用代码。

units = {
    "Bitcoin": "USD per BTC",
    "Ethereum": "USD per ETH",
    "SP500": "Index points",
    "Nasdaq100": "Index points",
    "CSI300": "Index points",
    "CSI1000": "Index points"
}
# 为每种资产设置纵坐标单位，避免把指数点位误认为美元价格。

periods = {
    "1_week": pd.DateOffset(weeks=1),
    "1_month": pd.DateOffset(months=1),
    "6_months": pd.DateOffset(months=6),
    "1_year": pd.DateOffset(years=1)
}
# 定义四个分析窗口，使用日历周、日历月和日历年计算起始日期。

annualization = {
    "Bitcoin": 365,
    "Ethereum": 365,
    "SP500": 252,
    "Nasdaq100": 252,
    "CSI300": 252,
    "CSI1000": 252
}
# 加密货币按照每年365天估算年化波动率。
# 股票指数按照每年约252个交易日估算年化波动率。

all_data = {}
# 创建空字典，用于保存成功下载的六种资产的数据。

used_tickers = {}
# 记录每种资产最终使用的 Yahoo Finance 代码。

In [7]:
# PART 3: 下载数据并保存 CSV

In [8]:
for asset_name, ticker_list in assets.items():
    # 遍历六种资产；asset_name 是名称，ticker_list 是候选代码列表。

    for ticker in ticker_list:
        # 按顺序尝试该资产对应的 Yahoo Finance 代码。

        try:
            # 使用 try 捕获网络错误或数据源错误，避免程序直接中断。

            df = yf.download(
                ticker,
                start=download_start.strftime("%Y-%m-%d"),
                end=download_end.strftime("%Y-%m-%d"),
                interval="1d",
                auto_adjust=False,
                multi_level_index=False,
                progress=False,
                threads=False
            )
            # 下载日度行情，保留 Close 列，不自动调整 OHLC 价格。
            # multi_level_index=False 使单只资产的数据表使用普通列名。
            # threads=False 关闭并行下载，便于逐个排查错误。

            if df.empty or "Close" not in df.columns:
                # 检查是否返回空数据，以及是否包含收盘价列。
                print(f"No valid data: {ticker}")
                # 打印无法获取有效数据的代码。
                continue
                # 当前代码无效，继续尝试下一个备用代码。

            df.index = pd.to_datetime(df.index).tz_localize(None).normalize()
            # 将日期索引转换为无时区的日历日期，方便不同市场对齐。

            df = df.sort_index()
            # 按日期从早到晚排序。

            df = df.loc[df.index >= start_date].copy()
            # 只保留最近一年的数据，不使用额外提前下载的七天。

            df = df.dropna(subset=["Close"])
            # 删除没有收盘价的日期，避免后续计算出现错误。

            df = df.loc[df["Close"] > 0].copy()
            # 排除无效的零价格或负价格。

            if len(df) < 2:
                # 至少需要两个价格观测值才能计算收益率。
                continue
                # 数据不足时继续尝试备用代码。

            all_data[asset_name] = df
            # 将有效 DataFrame 保存到 all_data 字典中。

            used_tickers[asset_name] = ticker
            # 记录成功使用的 Yahoo Finance 代码。

            df.to_csv(data_dir / f"{asset_name}.csv", index_label="Date")
            # 将完整的日度行情保存为 CSV，日期作为第一列。

            print(f"Downloaded: {asset_name} ({ticker}), {len(df)} rows")
            # 打印资产名称、代码以及成功下载的数据行数。

            break
            # 已成功下载该资产，不再尝试其他备用代码。

        except Exception as error:
            # 捕获当前代码下载或处理时发生的异常。
            print(f"Failed: {ticker} | {error}")
            # 打印失败原因，方便检查网络或代码问题。

    if asset_name not in all_data:
        # 检查所有候选代码是否都没有返回有效数据。
        print(f"WARNING: {asset_name} could not be downloaded.")
        # 明确警告该资产缺失，不用其他资产的数据替代。

Downloaded: Bitcoin (BTC-USD), 366 rows
Downloaded: Ethereum (ETH-USD), 366 rows
Downloaded: SP500 (^GSPC), 251 rows
Downloaded: Nasdaq100 (^NDX), 251 rows


In [9]:
# PART 4: 定义统计指标计算函数

In [10]:

def calculate_statistics(prices, trading_days):
    # 定义一个函数，输入价格序列和年化天数，返回统计指标。

    prices = prices.dropna()
    # 删除价格序列中的缺失值。

    if len(prices) < 3:
        # 少于三个价格观测值时，无法稳定估计样本标准差。
        return None
        # 返回 None，表示该窗口数据不足。

    first_price = float(prices.iloc[0])
    # 提取分析窗口中第一个有效收盘价。

    last_price = float(prices.iloc[-1])
    # 提取分析窗口中最后一个有效收盘价。

    change_pct = (last_price / first_price - 1) * 100
    # 计算期初到期末的累计价格变化百分比。

    daily_returns = prices.pct_change().dropna()
    # 计算相邻观测日之间的简单收益率，并删除第一个缺失值。

    daily_volatility = daily_returns.std(ddof=1)
    # 计算收益率的样本标准差，作为日度波动率估计。

    annual_volatility = daily_volatility * np.sqrt(trading_days) * 100
    # 根据平方根时间规则，将波动率年化并转换为百分数。

    return {
        "first_price": first_price,
        "end_price": last_price,
        "change_pct": change_pct,
        "annual_volatility_pct": annual_volatility,
        "observations": len(prices),
        "first_date": prices.index[0],
        "last_date": prices.index[-1]
    }
    # 返回一个字典，包含图表标注和 CSV 汇总所需的全部指标。

In [11]:
# PART 5: 绘制每种资产的四张价格折线图

In [12]:
summary_rows = []
# 创建空列表，用于收集每种资产在四个时间窗口的统计结果。

for asset_name, df in all_data.items():
    # 依次处理已经成功下载的资产。

    for period_name, offset in periods.items():
        # 对每种资产依次绘制一周、一月、半年和一年图表。

        cutoff = today - offset
        # 计算当前时间窗口的日历起始日期。

        window = df.loc[df.index >= cutoff, "Close"].dropna()
        # 选取起始日期之后的收盘价序列。

        stats = calculate_statistics(window, annualization[asset_name])
        # 计算该窗口的累计价格变化和年化历史波动率。

        if stats is None:
            # 检查当前窗口是否有足够的价格观测值。
            print(f"Not enough data: {asset_name}, {period_name}")
            # 打印数据不足的窗口名称。
            continue
            # 跳过当前窗口，继续处理其他窗口。

        fig, ax = plt.subplots(figsize=(11, 6))
        # 创建一张宽11英寸、高6英寸的 Figure 和 Axes。

        ax.plot(window.index, window.values, linewidth=2)
        # 以日期为横坐标、资产收盘价为纵坐标绘制折线。

        ax.set_title(f"{asset_name} | {period_name}", fontsize=15)
        # 设置图表标题，标明资产和分析期限。

        ax.set_xlabel("Date")
        # 设置横坐标标题为日期。

        ax.set_ylabel(units[asset_name])
        # 设置纵坐标单位，区分美元价格和指数点位。

        ax.grid(True, alpha=0.3)
        # 添加浅色网格，方便观察价格变化。

        annotation = (
            f"Start: {stats['first_price']:,.2f}\n"
            f"End: {stats['end_price']:,.2f}\n"
            f"Change: {stats['change_pct']:+.2f}%\n"
            f"Annualized volatility: {stats['annual_volatility_pct']:.2f}%"
        )
        # 生成统计说明，包括期初价格、期末价格、涨跌幅和年化波动率。

        ax.text(
            0.02, 0.98, annotation,
            transform=ax.transAxes,
            va="top",
            fontsize=10,
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="gray")
        )
        # 将统计说明放在图表左上角，并使用白色半透明背景。

        fig.autofmt_xdate()
        # 自动旋转和调整日期标签，避免横坐标文字重叠。

        fig.tight_layout()
        # 自动调整图表边距，减少标题和坐标标签被裁切的风险。

        fig.savefig(single_dir / f"{asset_name}_{period_name}.png", dpi=180)
        # 将当前图表保存为高清 PNG 图片。

        plt.close(fig)
        # 关闭已经保存的图表，避免28张图同时占用过多内存。

        summary_rows.append({
            "Asset": asset_name,
            "Ticker": used_tickers[asset_name],
            "Period": period_name,
            "First_Date": stats["first_date"].date(),
            "Last_Date": stats["last_date"].date(),
            "Start_Price": stats["first_price"],
            "End_Price": stats["end_price"],
            "Price_Change_Pct": stats["change_pct"],
            "Annualized_Volatility_Pct": stats["annual_volatility_pct"],
            "Observations": stats["observations"]
        })
        # 把本次计算结果保存到汇总列表，供后续导出 CSV。


In [13]:
# PART 6: 将六种资产的收盘价合并到一张数据表

In [14]:
close_series = []
# 创建空列表，用于保存每种资产的收盘价序列。

for asset_name, df in all_data.items():
    # 遍历成功下载的资产数据。

    series = df["Close"].rename(asset_name)
    # 取出收盘价，并将序列名称设置为资产名称。

    close_series.append(series)
    # 把该资产的收盘价序列加入列表。

if len(close_series) == 0:
    # 如果所有资产都下载失败，则无法继续生成比较图。
    raise RuntimeError("No asset data was downloaded successfully.")
    # 明确报错并停止程序。

combined = pd.concat(close_series, axis=1).sort_index()
# 按日期把各资产收盘价合并成一个 DataFrame。
# 不同市场休市日期不同，因此部分单元格可能为 NaN。

combined.index.name = "Date"
# 为合并后的日期索引设置名称。

combined.to_csv(data_dir / "all_assets_close_prices.csv")
# 保存六种资产的原始收盘价数据表。


In [15]:
# PART 7: 绘制四张归一化价格比较图

In [16]:
for period_name, offset in periods.items():
    # 分别生成最近一周、一月、半年和一年的比较图。

    cutoff = today - offset
    # 计算当前比较期限的日历起始日期。

    comparison = combined.loc[combined.index >= cutoff].copy()
    # 从合并数据表中截取当前时间窗口。

    comparison = comparison.dropna(axis=0, how="any")
    # 只保留所有成功下载的资产都有价格的共同日期。
    # 这样可以避免把不同日期的收盘价当成同一天比较。

    if len(comparison) < 2:
        # 检查共同日期是否至少有两个观测值。
        print(f"Not enough common dates: {period_name}")
        # 打印无法生成比较图的期限。
        continue
        # 跳过当前期限。

    normalized = comparison.div(comparison.iloc[0]).mul(100)
    # 每一列都除以该资产在共同起始日的价格，再乘以100。
    # 因此六种资产的第一天全部从100开始。

    normalized.index.name = "Date"
    # 为归一化数据设置日期索引名称。

    normalized.to_csv(data_dir / f"normalized_{period_name}.csv")
    # 将当前期限的归一化数据保存为 CSV。

    fig, ax = plt.subplots(figsize=(12, 7))
    # 创建一张用于比较多种资产走势的图表。

    for asset_name in normalized.columns:
        # 依次绘制每种资产的归一化价格曲线。

        series = normalized[asset_name]
        # 提取当前资产的归一化价格序列。

        total_change = series.iloc[-1] - 100
        # 因为期初为100，所以期末减100就是累计变动百分比数值。

        ax.plot(
            series.index,
            series.values,
            linewidth=2,
            label=f"{asset_name} ({total_change:+.2f}%)"
        )
        # 绘制当前资产的曲线，并在图例中注明该期限的累计涨跌幅。

    ax.axhline(100, linestyle="--", color="gray", alpha=0.7)
    # 在100的位置添加水平虚线，作为所有资产共同的期初基准。

    ax.set_title(f"Normalized Asset Performance | {period_name}", fontsize=15)
    # 设置比较图标题。

    ax.set_xlabel("Date")
    # 设置横坐标标题。

    ax.set_ylabel("Normalized Price (Start = 100)")
    # 设置纵坐标标题，说明所有资产期初统一为100。

    ax.legend(loc="best", fontsize=9)
    # 显示图例，包括各资产名称及期末累计涨跌幅。

    ax.grid(True, alpha=0.3)
    # 添加浅色网格，提高图表可读性。

    fig.autofmt_xdate()
    # 自动调整日期标签。

    fig.tight_layout()
    # 调整图表边距。

    fig.savefig(comparison_dir / f"comparison_{period_name}.png", dpi=180)
    # 将当前期限的归一化比较图保存为 PNG。

    plt.close(fig)
    # 关闭图表并释放内存。



In [17]:
# PART 8: 导出统计汇总并打印完成信息

In [18]:
summary_df = pd.DataFrame(summary_rows)
# 将之前收集的统计字典列表转换成 Pandas DataFrame。

summary_df.to_csv(output_dir / "summary_statistics.csv", index=False)
# 将各资产四个期限的涨跌幅和波动率保存为 CSV。

print("\nAnalysis completed.")
# 提示程序运行完成。

print(f"Output folder: {output_dir}")
# 打印输出文件夹的绝对路径，方便在电脑上查找。

print(f"Assets downloaded: {len(all_data)} / {len(assets)}")
# 打印成功下载的资产数量，检查是否全部六种资产都已获取。

print(f"Single-asset charts: {len(summary_rows)}")
# 打印成功生成的单资产图表数量。

print("\nSummary statistics:")
# 输出统计结果标题。

print(summary_df.to_string(index=False))
# 在控制台中显示各资产、各期限的统计结果。



Analysis completed.
Output folder: asset_analysis
Assets downloaded: 4 / 6
Single-asset charts: 16

Summary statistics:
    Asset  Ticker   Period First_Date  Last_Date   Start_Price    End_Price  Price_Change_Pct  Annualized_Volatility_Pct  Observations
  Bitcoin BTC-USD   1_week 2026-11-05 2026-11-12  75942.125000 73205.000000         -3.604225                  19.779843             8
  Bitcoin BTC-USD  1_month 2026-10-12 2026-11-12  81762.562500 73205.000000        -10.466358                  34.923597            32
  Bitcoin BTC-USD 6_months 2026-05-12 2026-11-12  80477.492188 73205.000000         -9.036678                  38.012695           185
  Bitcoin BTC-USD   1_year 2025-11-12 2026-11-12 101663.187500 73205.000000        -27.992618                  44.113795           366
 Ethereum ETH-USD   1_week 2026-11-05 2026-11-12   2234.476318  2037.689941         -8.806823                  43.207408             8
 Ethereum ETH-USD  1_month 2026-10-12 2026-11-12   2444.357422  2037.